# Análisis de la Brecha Digital en el Perú (Capa Gold)

Este notebook consume directamente el dataset analítico consolidado en la **Capa Gold** (`data/gold/brecha_digital_osiptel.parquet`).

### Principio Arquitectural de Big Data:
El notebook **no realiza ingesta, descargas ni limpiezas pesadas**. Toda la preparación y estandarización de datos (Raw y Silver) fue procesada de manera desacoplada por el orquestador `python run_pipeline.py`.

### Objetivos Analíticos:
1. Cargar el dataset final de la capa Gold (`data/gold/brecha_digital_osiptel.parquet`).
2. Calcular KPIs macro a nivel nacional (volumen de conexiones fijas y penetración de fibra óptica).
3. Analizar la evolución histórica trimestral y la transición tecnológica (Fibra vs. Cablemódem).
4. Evaluar la brecha digital departamental y la correlación con la competencia de operadoras.
5. Formular conclusiones ejecutivas basadas en evidencia para la toma de decisiones regulatorias.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de estilo visual para gráficos
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# 1. Cargar únicamente la capa Gold ya procesada por el pipeline
RUTAS_GOLD = [
    Path("../data/gold/brecha_digital_osiptel.parquet"),
    Path("data/gold/brecha_digital_osiptel.parquet"),
    Path("../data/gold/brecha_digital_consolidada.parquet"),
    Path("data/gold/brecha_digital_consolidada.parquet")
]

ruta_gold = next((p for p in RUTAS_GOLD if p.exists()), Path("../data/gold/brecha_digital_osiptel.parquet"))

try:
    df_gold = pd.read_parquet(ruta_gold)
    print(f"✅ Capa Gold cargada exitosamente desde: {ruta_gold}")
    print(f"Dimensiones analíticas: {df_gold.shape[0]} registros × {df_gold.shape[1]} columnas")
except Exception as e:
    print(f"⚠️ Archivo no encontrado en {ruta_gold}. Ejecuta primero: python run_pipeline.py")
    raise e

# 2. Vista preliminar de la capa analítica Gold
display(df_gold.head())

### 1. Estructura y Esquema del Dataset Gold

In [ ]:
df_gold.info()
display(df_gold.describe().T[["mean", "std", "min", "50%", "max"]])

### 2. KPIs Macro a Nivel Nacional

Calculamos los indicadores de volumen y penetración en el periodo más reciente registrado.

In [ ]:
max_anio = df_gold["anio"].astype(str).max()
ultimo_trim = df_gold[df_gold["anio"].astype(str) == max_anio]["trimestre"].max()

df_actual = df_gold[(df_gold["anio"].astype(str) == max_anio) & (df_gold["trimestre"] == ultimo_trim)]

total_conexiones = df_actual["total_conexiones"].sum()
total_fibra = df_actual["conexiones_fibra_optica"].sum()
pct_fibra = (total_fibra / total_conexiones) * 100 if total_conexiones > 0 else 0

print(f"=== INDICADORES NACIONALES ({max_anio} - Trimestre {ultimo_trim}) ===")
print(f"• Total Conexiones Fijas:       {total_conexiones:,.0f}")
print(f"• Conexiones en Fibra Óptica:   {total_fibra:,.0f}")
print(f"• Penetración de Fibra Óptica:  {pct_fibra:.2f}%")
print(f"• Departamentos Monitoreados:   {df_actual['departamento'].nunique()}")

### 3. Evolución Histórica Trimestral: Transición a Fibra Óptica

In [ ]:
df_evol = df_gold.groupby(["anio", "trimestre"]).agg({
    "total_conexiones": "sum",
    "conexiones_fibra_optica": "sum",
    "conexiones_cablemodem": "sum"
}).reset_index()

df_evol["periodo_label"] = df_evol["anio"].astype(str) + "-T" + df_evol["trimestre"].astype(str)
df_evol = df_evol.sort_values(by=["anio", "trimestre"])

plt.figure(figsize=(14, 6))
plt.plot(df_evol["periodo_label"], df_evol["total_conexiones"] / 1e6, marker="o", linewidth=2.5, label="Total Conexiones (M)", color="#1f77b4")
plt.plot(df_evol["periodo_label"], df_evol["conexiones_fibra_optica"] / 1e6, marker="s", linewidth=2.5, label="Fibra Óptica (M)", color="#2ca02c")
plt.plot(df_evol["periodo_label"], df_evol["conexiones_cablemodem"] / 1e6, marker="^", linewidth=2, linestyle="--", label="Cablemódem (M)", color="#ff7f0e")

plt.title("Evolución Trimestral de Conexiones Fijas por Tecnología en Perú", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Periodo Trimestral", fontsize=12)
plt.ylabel("Millones de Conexiones", fontsize=12)
plt.xticks(rotation=45)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

### 4. Brecha Digital Departamental (Ranking Territorial)

In [ ]:
ranking_dept = df_actual.groupby("departamento").agg({
    "total_conexiones": "sum",
    "conexiones_fibra_optica": "sum",
    "total_operadoras_activas": "mean"
}).reset_index()

ranking_dept["pct_fibra"] = (ranking_dept["conexiones_fibra_optica"] / ranking_dept["total_conexiones"]) * 100
ranking_dept = ranking_dept.sort_values(by="total_conexiones", ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=ranking_dept, x="total_conexiones", y="departamento", palette="viridis")
plt.title(f"Distribución de Conexiones de Internet Fijo por Departamento ({max_anio}-T{ultimo_trim})", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Total de Conexiones Activas", fontsize=12)
plt.ylabel("Departamento", fontsize=12)
plt.gca().xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{int(x):,}"))
plt.tight_layout()
plt.show()

### 5. Competencia de Mercado y Penetración de Fibra Óptica

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=ranking_dept,
    x="total_operadoras_activas",
    y="pct_fibra",
    size="total_conexiones",
    sizes=(100, 1000),
    hue="pct_fibra",
    palette="mako",
    legend=None
)

for _, row in ranking_dept.iterrows():
    plt.annotate(
        row["departamento"],
        (row["total_operadoras_activas"], row["pct_fibra"]),
        fontsize=8,
        xytext=(5, 2),
        textcoords="offset points"
    )

plt.title("Relación entre Variedad de Operadoras y Adopción de Fibra Óptica", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Número de Operadoras Activas en el Departamento", fontsize=12)
plt.ylabel("% Adopción de Fibra Óptica", fontsize=12)
plt.tight_layout()
plt.show()

### 6. Conclusiones y Hallazgos Analíticos para OSIPTEL

1. **Transición Tecnológica hacia Fibra Óptica:** La curva de crecimiento de fibra óptica (FTTH) refleja un proceso de sustitución acelerada frente a tecnologías convencionales como cablemódem y par de cobre.
2. **Concentración Geográfica y Brecha Digital:** La concentración del volumen de conexiones fijas se localiza predominantemente en Lima y regiones costeras (Arequipa, La Libertad), mientras que departamentos de sierra central y selva registran menor densidad de accesos.
3. **Impacto Regulatorio y Competencia:** Los departamentos con mayor dinamismo competitivo y mayor cantidad de operadoras activas muestran una adopción proporcionalmente mayor de fibra óptica, destacando la relevancia de las políticas de fomento a la infraestructura de telecomunicaciones.